# 🤖 Step 3 — PhoBERT Fine-tuning for Multi-task ABSA

**Mục tiêu:**
- Fine-tune `vinai/phobert-base` với shared encoder + N classification heads
- Class-weighted CrossEntropy (từ `class_weights.json`)
- Early-stopping theo Macro-F1 val
- So sánh với TF-IDF baseline và BiLSTM

> Cài đặt: `pip install transformers torch accelerate`

In [ ]:
import os, json, pathlib, random, warnings, gc
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid')
print(f"✅ PyTorch {torch.__version__} | Device: {DEVICE}")

hf_logging.set_verbosity_warning()

c:\Users\annt43\Desktop\finalProject\venv_tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch 2.11.0+cpu | Device: cpu


## 1. Load processed data

In [4]:
processed_dir = pathlib.Path('../data/processed')
train = pd.read_parquet(processed_dir / 'train.parquet')
val   = pd.read_parquet(processed_dir / 'val.parquet')
test  = pd.read_parquet(processed_dir / 'test.parquet')

with open(processed_dir / 'metadata.json', encoding='utf-8') as fh:
    meta = json.load(fh)
with open(processed_dir / 'class_weights.json', encoding='utf-8') as fh:
    class_weights_dict = json.load(fh)

aspect_cols = meta['aspect_cols']
text_col = 'text_tokenized' if 'text_tokenized' in train.columns else 'text_clean'

NONE_LABEL = '__NONE__'
LABELS = [NONE_LABEL, 'positive', 'negative', 'neutral']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_CLASSES = len(LABELS)

def to_label_ids(df):
    Y = df[aspect_cols].fillna(NONE_LABEL).astype(str)
    Y = Y.map(lambda s: LABEL2ID.get(s, LABEL2ID[NONE_LABEL]))
    return Y.values.astype(np.int64)

Y_tr = to_label_ids(train)
Y_vl = to_label_ids(val)
Y_te = to_label_ids(test)

X_tr = train[text_col].fillna('').astype(str).tolist()
X_vl = val[text_col].fillna('').astype(str).tolist()
X_te = test[text_col].fillna('').astype(str).tolist()

print(f"Train/Val/Test: {len(X_tr)} / {len(X_vl)} / {len(X_te)}")
print(f"Aspects: {len(aspect_cols)}  |  Labels: {LABELS}")

Train/Val/Test: 2032 / 436 / 436
Aspects: 12  |  Labels: ['__NONE__', 'positive', 'negative', 'neutral']


## 2. PhoBERT Tokenizer

### Nếu chưa có model local
Chạy trong terminal (thư mục notebooks):

`pip install -U "huggingface_hub[cli]"`
`huggingface-cli download vinai/phobert-base --local-dir ../models/phobert-base --local-dir-use-symlinks False`

Sau đó chạy lại từ Cell 6.

In [13]:
import os, certifi, pathlib
from transformers import PhobertTokenizer

os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()

MODEL_NAME = 'vinai/phobert-base'
MAX_LEN = 256
CACHE_DIR = pathlib.Path('../models/hf_cache')
LOCAL_MODEL_DIR = pathlib.Path('../models/phobert-base')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Required model files
required_any = [LOCAL_MODEL_DIR / 'model.safetensors', LOCAL_MODEL_DIR / 'pytorch_model.bin']
required_all = [LOCAL_MODEL_DIR / 'config.json']

# Required tokenizer files for PhoBERT
tokenizer_merges = LOCAL_MODEL_DIR / 'bpe.codes'
tokenizer_vocab_candidates = [LOCAL_MODEL_DIR / 'dict.txt', LOCAL_MODEL_DIR / 'vocab.txt']
tokenizer_vocab = next((p for p in tokenizer_vocab_candidates if p.exists()), None)

missing_all = [p.name for p in required_all if not p.exists()]
has_weight = any(p.exists() for p in required_any)

if missing_all or not has_weight or (not tokenizer_merges.exists()) or (tokenizer_vocab is None):
    raise RuntimeError(
        'Thiếu model/tokenizer local tại ../models/phobert-base.\n'
        'PhoBERT cần tối thiểu: config.json + (model.safetensors|pytorch_model.bin) + bpe.codes + (dict.txt|vocab.txt).\n'
        'Tải lại đầy đủ:\n'
        'huggingface-cli download vinai/phobert-base --local-dir ../models/phobert-base --local-dir-use-symlinks False'
    )

MODEL_SOURCE = str(LOCAL_MODEL_DIR)
LOCAL_ONLY = True

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_SOURCE,
        cache_dir=str(CACHE_DIR),
        use_fast=False,
        local_files_only=LOCAL_ONLY
    )
except Exception as e:
    print(f'⚠️ AutoTokenizer lỗi: {type(e).__name__}: {e}')
    print('↪ Fallback sang PhobertTokenizer với file local tường minh...')
    tokenizer = PhobertTokenizer(
        vocab_file=str(tokenizer_vocab),
        merges_file=str(tokenizer_merges)
    )

print(f'✅ Tokenizer loaded from local: {MODEL_SOURCE} | vocab={tokenizer_vocab.name} | merges={tokenizer_merges.name}')

# Quick check
sample_enc = tokenizer(X_tr[0], max_length=MAX_LEN, truncation=True, padding='max_length')
print(f"Sample input_ids length: {len(sample_enc['input_ids'])}")

✅ Tokenizer loaded from local: ..\models\phobert-base | vocab=vocab.txt | merges=bpe.codes
Sample input_ids length: 256


## 3. Dataset & DataLoader

In [14]:
class ABSADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

BATCH_SIZE = 16

ds_train = ABSADataset(X_tr, Y_tr, tokenizer)
ds_val   = ABSADataset(X_vl, Y_vl, tokenizer)
ds_test  = ABSADataset(X_te, Y_te, tokenizer)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

print(f"Batches — train: {len(dl_train)}, val: {len(dl_val)}, test: {len(dl_test)}")

Batches — train: 127, val: 14, test: 14


## 4. Model — PhoBERT + Multi-head Classification

In [ ]:
class PhoBERTMultiHead(nn.Module):
    def __init__(self, model_source, n_aspects, n_classes, dropout=0.1, cache_dir=None, local_only=True):
        super().__init__()
        self.n_aspects = n_aspects
        self.n_classes = n_classes

        bert_model, loading_info = AutoModel.from_pretrained(
            model_source,
            cache_dir=cache_dir,
            local_files_only=local_only,
            output_loading_info=True
        )
        self.bert = bert_model

        unexpected = sorted(loading_info.get('unexpected_keys', []))
        bad_unexpected = [k for k in unexpected if not k.startswith('lm_head.')]
        if bad_unexpected:
            raise RuntimeError(f'Unexpected keys không hợp lệ: {bad_unexpected[:10]}')
        if unexpected:
            print(f"ℹ️ Ignored {len(unexpected)} unexpected keys (MLM head): {unexpected[:3]}...")

        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(hidden, n_classes) for _ in range(n_aspects)])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        cls = self.dropout(out.last_hidden_state[:, 0])
        logits = [head(cls) for head in self.heads]
        return logits

model = PhoBERTMultiHead(
    MODEL_SOURCE,
    len(aspect_cols),
    N_CLASSES,
    dropout=0.1,
    cache_dir=str(CACHE_DIR),
    local_only=True
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model params: {total_params:,} total | {train_params:,} trainable")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13964.64it/s]
[transformers] RobertaModel LOAD REPORT from: ..\models\phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13964.64it/s]
[transformers] RobertaModel LOAD REPORT from: ..\models\phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_no

✅ Model params: 135,035,184 total | 135,035,184 trainable


In [17]:
# Smoke test output (Section 4)
model.eval()
with torch.no_grad():
    b = next(iter(dl_train))
    input_ids = b['input_ids'][:2].to(DEVICE)
    attention_mask = b['attention_mask'][:2].to(DEVICE)
    token_type_ids = b['token_type_ids'][:2].to(DEVICE) if 'token_type_ids' in b else None

    logits_list = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

    print(f'num_heads={len(logits_list)} | expected={len(aspect_cols)}')
    assert len(logits_list) == len(aspect_cols), 'Sai số lượng heads/output'
    for i, lg in enumerate(logits_list[:3]):
        print(f'head[{i}] shape={tuple(lg.shape)} | finite={torch.isfinite(lg).all().item()}')
        assert lg.shape == (input_ids.size(0), N_CLASSES), 'Sai shape logits'
        assert torch.isfinite(lg).all(), 'Logits chứa NaN/Inf'

print('✅ Section 4 output looks correct: số head + shape + giá trị logits đều hợp lệ.')

c:\Users\annt43\Desktop\finalProject\venv_tf\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


num_heads=12 | expected=12
head[0] shape=(2, 4) | finite=True
head[1] shape=(2, 4) | finite=True
head[2] shape=(2, 4) | finite=True
✅ Section 4 output looks correct: số head + shape + giá trị logits đều hợp lệ.


## 5. Class weights + Loss

In [ ]:
# Build per-aspect class weight tensors
aspect_loss_fns = []
for asp in aspect_cols:
    w = torch.ones(N_CLASSES, dtype=torch.float32)
    cw = class_weights_dict.get(asp, {})
    for lab, idx in LABEL2ID.items():
        if lab in cw:
            w[idx] = float(cw[lab])
    aspect_loss_fns.append(
        nn.CrossEntropyLoss(weight=w.to(DEVICE))
    )
print(f"✅ {len(aspect_loss_fns)} weighted CrossEntropy losses created")

## 6. Training Loop

In [ ]:
EPOCHS = 10
LR = 2e-5
WARMUP_RATIO = 0.1
PATIENCE = 3
ACCUMULATION_STEPS = 2  # gradient accumulation

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(dl_train) * EPOCHS // ACCUMULATION_STEPS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

def compute_loss(logits_list, labels):
    """labels: (B, n_aspects)"""
    loss = 0.0
    for i, (logits, loss_fn) in enumerate(zip(logits_list, aspect_loss_fns)):
        loss += loss_fn(logits, labels[:, i])
    return loss / len(aspect_cols)

@torch.no_grad()
def evaluate_model(model, dl):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dl:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels']
        logits_list = model(input_ids, attention_mask)
        preds = np.stack([l.argmax(-1).cpu().numpy() for l in logits_list], axis=1)
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    P = np.concatenate(all_preds)
    Y = np.concatenate(all_labels)
    per_f1 = [f1_score(Y[:, i], P[:, i], average='macro', zero_division=0)
              for i in range(len(aspect_cols))]
    return float(np.mean(per_f1)), per_f1, Y, P

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
best_f1 = -1.0
best_weights = None
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(dl_train, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        logits_list = model(input_ids, attention_mask)
        loss = compute_loss(logits_list, labels) / ACCUMULATION_STEPS
        loss.backward()

        if (step + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix(loss=f'{loss.item() * ACCUMULATION_STEPS:.4f}')

    train_loss = running_loss / len(dl_train)
    val_f1, per_f1, _, _ = evaluate_model(model, dl_val)

    history['train_loss'].append(train_loss)
    history['val_macro_f1'].append(val_f1)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch:02d} | loss={train_loss:.4f} | val_macro_f1={val_f1:.4f} | lr={lr_now:.2e}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
        print(f'  ✅ New best!')
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'  ⏹ Early stopping at epoch {epoch}')
            break

# Restore best
if best_weights:
    model.load_state_dict(best_weights)
    model.to(DEVICE)
print(f'\n🏆 Best val Macro-F1: {best_f1:.4f}')

In [ ]:
# Plot history
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history['train_loss'], marker='o', color='#EF5350', label='train')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['val_macro_f1'], marker='o', color='#26A69A')
ax[1].set_title('Val Macro-F1'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 7. Đánh giá trên Test

In [ ]:
test_f1, per_aspect_f1, Yt, Pt = evaluate_model(model, dl_test)
print(f'🧪 Test Macro-F1 (mean across aspects): {test_f1:.4f}')

per_asp = pd.DataFrame({'aspect': aspect_cols, 'macro_f1': per_aspect_f1}).set_index('aspect').round(4)
display(per_asp)

# Top-3 aspects by support
support = (Y_tr != LABEL2ID[NONE_LABEL]).sum(axis=0)
top_idx = np.argsort(-support)[:3]
for i in top_idx:
    asp = aspect_cols[i]
    print(f"\n━━━ {asp} ━━━")
    y_true = [ID2LABEL[v] for v in Yt[:, i]]
    y_pred = [ID2LABEL[v] for v in Pt[:, i]]
    print(classification_report(y_true, y_pred, zero_division=0))

## 8. So sánh với Baseline TF-IDF và BiLSTM

In [ ]:
out_dir = pathlib.Path('../outputs/phobert')
out_dir.mkdir(parents=True, exist_ok=True)

comp = per_asp.rename(columns={'macro_f1': 'phobert_f1'})

# Load TF-IDF baseline
tfidf_csv = pathlib.Path('../outputs/baseline/test_metrics.csv')
if tfidf_csv.exists():
    base = pd.read_csv(tfidf_csv, index_col=0)
    comp = comp.join(base[['macro_f1']].rename(columns={'macro_f1': 'tfidf_f1'}), how='left')

# Load BiLSTM
bilstm_csv = pathlib.Path('../outputs/neural_bilstm/test_metrics.csv')
if bilstm_csv.exists():
    bilstm = pd.read_csv(bilstm_csv, index_col=0)
    comp = comp.join(bilstm[['macro_f1']].rename(columns={'macro_f1': 'bilstm_f1'}), how='left')

print('=== Test Macro-F1: All Models ===')
display(comp.round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(aspect_cols)), 5))
x = np.arange(len(aspect_cols)); w = 0.25
colors = {'tfidf_f1': '#90A4AE', 'bilstm_f1': '#7E57C2', 'phobert_f1': '#EF5350'}
offset = -w
for col, color in colors.items():
    if col in comp.columns:
        ax.bar(x + offset, comp[col].fillna(0), w, label=col.replace('_f1','').upper(), color=color)
        offset += w
ax.set_xticks(x); ax.set_xticklabels(aspect_cols, rotation=45, ha='right')
ax.set_ylabel('Macro-F1 (test)'); ax.set_title('Model Comparison — Test Macro-F1')
ax.legend(); plt.tight_layout()
plt.savefig(out_dir / 'compare_all_models.png', dpi=150)
plt.show()

## 9. Lưu artifacts

In [ ]:
# Save model weights
torch.save(best_weights, out_dir / 'phobert_multihead.pt')

# Save config
per_asp_save = per_asp.rename(columns={'macro_f1': 'macro_f1'})
per_asp_save.to_csv(out_dir / 'test_metrics.csv')

summary = {
    'model_name':    MODEL_NAME,
    'framework':     'pytorch',
    'torch_version': torch.__version__,
    'max_len':       MAX_LEN,
    'batch_size':    BATCH_SIZE,
    'accumulation':  ACCUMULATION_STEPS,
    'lr':            LR,
    'epochs_run':    len(history['train_loss']),
    'n_aspects':     len(aspect_cols),
    'val_macro_f1':  float(best_f1),
    'test_macro_f1': float(test_f1),
    'per_aspect_f1': {asp: float(f) for asp, f in zip(aspect_cols, per_aspect_f1)},
    'history':       history,
}
with open(out_dir / 'summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2, default=str)

print(f"✅ Saved to {out_dir.resolve()}")
for p in sorted(out_dir.iterdir()):
    print(f"   • {p.name}")

---
## ✅ Checklist Step 3

- [x] Fine-tune PhoBERT (`vinai/phobert-base`) với multi-head classification
- [x] Class-weighted CrossEntropy
- [x] Gradient accumulation + linear warmup scheduler
- [x] Early-stopping theo Macro-F1 val
- [x] So sánh TF-IDF vs BiLSTM vs PhoBERT (test)
- [x] Lưu model weights (`.pt`), metrics, summary

**Bước tiếp theo:** Step 4 — Error Analysis & Ensemble → `notebooks/06_Error_Analysis.ipynb`